# Global Analysis of All Reconstructed Matrices

In [164]:
from pathlib import Path
import warnings
import networkx as nx
import json
import numpy as np
import pandas as pd
import torch

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


# Step 1: Load Full Reconstructed Dataset (Local PC)
Load reconstructed matrices.

In [165]:
WINDOW_LENGTH = 252
STRIDE = 5
FORWARD_DAYS = 252
FILE_NAME = 'data_00_20'
DATASET_NAME = f'{FILE_NAME}_w{WINDOW_LENGTH}_s{STRIDE}'
RUN_NAME = 'PCA_150dim'
DATASET = 'val'  # 'train', 'val', 'test' or 'all'
FILE_NAME = f'{DATASET}_reconstructed_{RUN_NAME}.pt'

project_root = Path.cwd().resolve().parent
dataset_dir = project_root / 'models' / DATASET_NAME / f'{FORWARD_DAYS}_days_gap' / 'PCA' / RUN_NAME / 'analysis_outputs'
ALL_RECONSTRUCTED_MATRIX_FILE = dataset_dir / f'{DATASET}_reconstructed_{RUN_NAME}.pt'
RESULTS_FOLDER = project_root / 'results' / DATASET_NAME / f'{FORWARD_DAYS}_days_gap' / f'reconstruction_analysis_{DATASET}' / RUN_NAME
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)


print(f'Dataset selected: {DATASET_NAME}')
print(f'All Reconstructed Matrix: {ALL_RECONSTRUCTED_MATRIX_FILE.name}')

Dataset selected: data_00_20_w252_s5
All Reconstructed Matrix: val_reconstructed_PCA_150dim.pt


In [166]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')
    corr_tensor = payload.get('corr_tensor', None)
    recon_tensor = payload.get('corr_tensor_reconstructed', None)
    indices = payload.get('indices', None)
    meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')
    if recon_tensor is None:
        raise KeyError('corr_tensor_reconstructed key not found in .pt file')
    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')
    if recon_tensor.ndim != 3 or recon_tensor.shape[1] != recon_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor_reconstructed: {recon_tensor.shape}')

    return corr_tensor.float(), recon_tensor.float(), indices, meta


gt_corr, recon_corr, indices, meta = load_corr_payload(ALL_RECONSTRUCTED_MATRIX_FILE)

print(f'Ground truth correlation tensor shape: {gt_corr.shape}')
print(f'Reconstructed correlation tensor shape: {recon_corr.shape}')
print(f'Indices: {indices[:10]}')
print(f'Metadata keys: {list(meta.keys())}')
print(gt_corr[:1,:5,:5])
print(recon_corr[:1,:5,:5])

num_windows, num_assets, _ = gt_corr.shape
print(f'\nNumber of windows: {num_windows}, Number of assets: {num_assets}')


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\dylan\\Desktop\\TESI\\SyntheticCorrelationVAE\\PROGETTO_TESI_DYLAN\\models\\data_00_20_w252_s5\\252_days_gap\\PCA\\PCA_150dim\\analysis_outputs\\val_reconstructed_PCA_150dim.pt'

# Errors Statistics (MSE,MAE,Frobenius on full Reconstructed Dataset)

In [ ]:
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    diff = original - reconstructed

    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [ ]:
gt_corr_np = gt_corr.cpu().numpy()
recon_corr_np = recon_corr.cpu().numpy()

errors_df, errors_stats = reconstruction_errors(gt_corr_np, recon_corr_np)

print(f'Reconstruction error statistics ({DATASET} dataset):')
display(errors_stats)
# 7. Preparazione payload JSON
file_path = RESULTS_FOLDER / f'reconstruction_errors_{DATASET}.json'
# Inverti righe e colonne con .T
errors_stats_dict = errors_stats.T.to_dict()

Reconstruction error statistics (val dataset):


,mean,std,min,median,max
MSE,0.007580,0.002042,0.004854,0.006757,0.011807
MAE,0.067463,0.008337,0.054712,0.064961,0.082912
Frobenius,8.632507,1.134890,6.967053,8.219842,10.865772


In [ ]:
def mst_edge_set(mst) -> set:
    return {frozenset(edge) for edge in mst.edges()}

def degree_distribution(mst, n_assets: int) -> np.ndarray:
    degrees = np.array([deg for _, deg in mst.degree()], dtype=int)
    counts = np.bincount(degrees, minlength=n_assets)
    return counts / counts.sum()

def compare_mst_metrics(mst_orig, mst_recon, n_assets: int, top_k: int):
    # Edges overlap and Jaccard
    edges_orig = mst_edge_set(mst_orig)
    edges_recon = mst_edge_set(mst_recon)
    common_edges = len(edges_orig & edges_recon)
    total_edges = max(n_assets - 1, 1)
    edge_overlap_pct = 100.0 * common_edges / total_edges
    edge_jaccard_pct = 100.0 * common_edges / max(len(edges_orig | edges_recon), 1)

    # Degree distribution and L1 distance
    dist_orig = degree_distribution(mst_orig, n_assets)
    dist_recon = degree_distribution(mst_recon, n_assets)
    degree_l1 = float(np.sum(np.abs(dist_orig - dist_recon)))

    # Average path length (weighted by distance)
    avg_path_len = nx.average_shortest_path_length(mst_orig, weight='weight')
    avg_path_len_recon = nx.average_shortest_path_length(mst_recon, weight='weight')
    avg_path_len_diff = float(abs(avg_path_len - avg_path_len_recon))

    # Average path length (unweighted, treating all edges as length 1)
    avg_path_len_unw = nx.average_shortest_path_length(mst_orig)
    avg_path_len_unw_recon = nx.average_shortest_path_length(mst_recon)
    avg_path_len_unw_diff = float(abs(avg_path_len_unw - avg_path_len_unw_recon))

    # Betweenness centrality top-k overlap
    top_k = int(min(top_k, n_assets))
    bet_orig = nx.betweenness_centrality(mst_orig, weight='weight', normalized=True)
    bet_recon = nx.betweenness_centrality(mst_recon, weight='weight', normalized=True)
    top_orig = {n for n, _ in sorted(bet_orig.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    top_recon = {n for n, _ in sorted(bet_recon.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    bet_overlap_pct = 100.0 * len(top_orig & top_recon) / max(top_k, 1)

    return {
        'edge_overlap_pct': edge_overlap_pct,
        'edge_jaccard_pct': edge_jaccard_pct,
        'degree_l1': degree_l1,
        'avg_path_len': float(avg_path_len),
        'avg_path_len_recon': float(avg_path_len_recon),
        'avg_path_len_diff': avg_path_len_diff,
        'avg_path_len_unweighted': float(avg_path_len_unw),
        'avg_path_len_unweighted_recon': float(avg_path_len_unw_recon),
        'avg_path_len_unweighted_diff': avg_path_len_unw_diff,
        'betweenness_topk_overlap_pct': bet_overlap_pct,
    }

In [ ]:
def sanitize_correlation_matrix(corr_matrix):
    corr_matrix = np.clip(corr_matrix, -1.0, 1.0)
    corr_matrix = (corr_matrix + corr_matrix.T) / 2.0
    np.fill_diagonal(corr_matrix, 1.0)
    
    return corr_matrix

def build_distance_matrix(corr_matrix):
    dist_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_matrix)))
    dist_matrix = (dist_matrix + dist_matrix.T) / 2.0
    np.fill_diagonal(dist_matrix, 0.0)
    
    return dist_matrix

def corr_to_mst(corr_matrix):
    """
    Converte una matrice di correlazione in un oggetto MST di NetworkX.
    Usa la metrica di distanza d = sqrt(2 * (1 - rho))
    """
    # 1. Calcolo della matrice delle distanze
    corr_matrix = sanitize_correlation_matrix(corr_matrix)
    dist_matrix = build_distance_matrix(corr_matrix)
    
    # 2. Creazione del grafo completo
    G = nx.from_numpy_array(dist_matrix)
    
    # 3. Calcolo del Minimum Spanning Tree
    mst = nx.minimum_spanning_tree(G, weight='weight')
    
    return mst

In [ ]:
# Inizializziamo una lista per contenere i risultati di ogni coppia di matrici
all_metrics = []

n_samples = gt_corr_np.shape[0]
n_assets = gt_corr_np.shape[1]
top_k_centrality = 10

print(f"Inizio elaborazione di {n_samples} matrici...")

for i in range(n_samples):
    # 1. Estrazione della singola matrice (2D)
    curr_gt = gt_corr_np[i]
    curr_recon = recon_corr_np[i]
    
    # 2. Generazione degli MST per questa istanza
    mst_gt = corr_to_mst(curr_gt)
    mst_recon = corr_to_mst(curr_recon)
    
    # 3. Calcolo metriche
    metrics = compare_mst_metrics(
        mst_gt, 
        mst_recon, 
        n_assets=n_assets, 
        top_k=top_k_centrality
    )
    
    # Aggiungiamo alla lista
    all_metrics.append(metrics)
    
    # Feedback ogni 50 iterazioni per monitorare il progresso
    if (i + 1) % 10 == 0:
        print(f"Elaborate {i + 1}/{n_samples} matrici...")

# 4. Creazione DataFrame e calcolo della media
results_df = pd.DataFrame(all_metrics)

# 5. Calcolo delle statistiche aggregate
stats_df = results_df.describe().T 

# 6. Conversione in un dizionario strutturato
# 'index' orient crea un dizionario dove le chiavi sono le metriche
stats_dict = stats_df.to_dict(orient='index')

# 7. Salvataggio unico di tutte le metriche
combined_metrics = {
    'reconstruction_errors': errors_stats_dict,
    'mst_metrics': stats_dict,
}

with open(file_path, 'w') as f:
    json.dump(combined_metrics, f, indent=4)

print(f"Statistiche salvate con successo in: {file_path}")

# --- Visualizzazione rapida a schermo delle statistiche aggregate ---
print(f"\n--- Summary Statistics ({DATASET} dataset) ---")
display(stats_df)

Inizio elaborazione di 131 matrici...
Elaborate 10/131 matrici...
Elaborate 20/131 matrici...
Elaborate 30/131 matrici...
Elaborate 40/131 matrici...
Elaborate 50/131 matrici...
Elaborate 60/131 matrici...
Elaborate 70/131 matrici...
Elaborate 80/131 matrici...
Elaborate 90/131 matrici...
Elaborate 100/131 matrici...
Elaborate 110/131 matrici...
Elaborate 120/131 matrici...
Elaborate 130/131 matrici...
Statistiche salvate con successo in: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\results\data_00_20_w252_s5\252_days_gap\reconstruction_analysis_val\PCA_100dim\reconstruction_errors_val.json

--- Summary Statistics (val dataset) ---


,count,mean,std,min,25%,50%,75%,max
edge_overlap_pct,131.0,26.933457,2.942554,20.202020,25.252525,27.272727,29.292929,33.333333
edge_jaccard_pct,131.0,15.595428,1.952787,11.235955,14.450867,15.789474,17.159763,20.000000
degree_l1,131.0,0.197252,0.054843,0.040000,0.160000,0.200000,0.240000,0.320000
avg_path_len,131.0,5.191302,0.883443,3.430791,4.679842,5.008442,5.692186,7.490088
avg_path_len_recon,131.0,5.218236,0.521129,4.031489,4.861071,5.246580,5.615308,6.351111
avg_path_len_diff,131.0,0.599958,0.552678,0.001830,0.206884,0.413755,0.828490,2.389166
avg_path_len_unweighted,131.0,6.604463,1.154710,4.503030,5.766162,6.385859,7.561111,9.870101
avg_path_len_unweighted_recon,131.0,6.019328,0.526625,4.880404,5.693030,5.978990,6.296061,7.336970
avg_path_len_unweighted_diff,131.0,0.958574,0.885365,0.032121,0.223232,0.669697,1.495152,3.860606
betweenness_topk_overlap_pct,131.0,48.091603,13.422971,20.000000,40.000000,50.000000,60.000000,90.000000
